<img src="https://www.hazfundacion.org/wp-content/uploads/2022/11/U-Carlos-III.png" width="300" alt="Universidad Carlos III de Madrid">

**Máster Universitario en Robótica y Automatización**

Asignatura: Procesamiento de Imágenes por Computador

Curso: 2026-27

Autores:
- Anthony Manzueta Difo
- Stefan Oshchypok
- Marina Ruiz Izu

Proyecto: **Reconocimiento de ocupación de plazas de aparcamiento mediante fine-tuning de redes preentrenadas**

# Problema a tratar

## Motivación

Conocer la disponibilidad de plazas libres es útil para la gestión de los aparcamientos y para reducir el tiempo y el tráfico que genera la búsqueda de plaza. Con la ocupación en tiempo real, los conductores ahorran tiempo de búsqueda.

Desde el punto de vista empresarial, la visión por computador aprovecha las cámaras de seguridad ya instaladas. Evita el coste de un sensor por plaza (en torno a 300-600 $ por plaza en los sistemas tradicionales) y la obra asociada. Además de la ocupación en tiempo real, permite obtener datos de uso (ocupación por franja horaria, tiempos de estancia) que ayudan a dimensionar y optimizar el aparcamiento. En empresas con parking en sus oficinas, mejora la experiencia de los empleados y reduce el tiempo y las emisiones de la búsqueda de plaza.

## Objetivo

Detectar plazas de aparcamiento libres y ocupadas mediante una cámara exterior. Dada una imagen de aparcamiento, el sistema localiza cada plaza y la clasifica como libre u ocupada, devolviendo para cada una un valor de confianza.

# Etapa 1: Preparación del Dataset

Aqui vamos a procesar un dataset [MetaPKLot-Dataset](https://huggingface.co/datasets/RajChauhan2911/MetaPKLot-Dataset) para crear nuestro mas pequeño y adaptado al problema enforntado.

El nuevo dataset creado debe ser subido al nuevo repositorio de [HuggingFace](https://huggingface.co/datasets/StafLoker/Meta-PKLot-mini).

## Imports

In [ ]:
!pip install -q huggingface_hub

import glob
import json
import os
from collections import Counter

from huggingface_hub import snapshot_download

## Config

In [ ]:
CONFIG = {
    'dataset': {
        'source': {
            'repo': 'RajChauhan2911/MetaPKLot-Dataset',
            'path': '/content/data/MetaPKLot',
        },
        'mini': {
            'repo': 'StafLoker/Meta-PKLot-mini',
            'path': '/content/data/Meta-PKLot-mini',
        },
    }
}

## Dataset

### Download

In [ ]:
snapshot_download(
    repo_id=CONFIG['dataset']['source']['repo'],
    repo_type='dataset',
    local_dir=CONFIG['dataset']['source']['path'],
    token=os.environ.get('HF_TOKEN'),
)

source_path = CONFIG['dataset']['source']['path']
num_files = 0
total_bytes = 0

for root, _, files in os.walk(source_path):
    num_files += len(files)
    total_bytes += sum(os.path.getsize(os.path.join(root, f)) for f in files)

print(f"Dataset size: {total_bytes / 1024**3:.2f} GB ({num_files} files)")

### Estructura de las anotaciones del dataset original

#### COCO

Las anotaciones están en formato **COCO** (JSON). Un fichero COCO tiene cinco secciones principales:

```json
{
  "info": {}, // Información del dataset: versión, descripción, fecha, etc.
  "licenses": [], // Licencias de las imágenes.
  "images": [], // Información de cada imagen.
  "categories": [], // Clases de objetos que reconoce el modelo.
  "annotations": [] // Anotaciones de los objetos presentes en las imágenes.
}
```

#### MetaPKLot

MetaPKLot sigue esta estructura y la reparte en `annotations/` en ficheros de vehículos (*cars*) y de plazas (*spots*). Ejemplo corto de un fichero de plazas:

```json
{
  "categories": [{"id": 0, "name": "empty"}, {"id": 1, "name": "occupied"}], // Clases de la plaza
  "climates": [{"id": 2, "name": "sunny"}], // Extra de MetaPKLot: climas posibles
  "images": [{
    "id": 0,
    "file_name": "UFPR04/Sunny/2012-12-07/2012-12-07_20_17_28.jpg", // Ruta relativa a la raíz del dataset
    "width": 1280,
    "height": 720,
    "annotationsRectangle": [x, y, w, h], // Región con anotación completa: fuera de ella no hay negativos
    "climate": 2,
    "dataset": "pklot",
    "subset": "ufpr04"
  }],
  "annotations": [{
    "id": 0,
    "image_id": 0, // Imagen a la que pertenece
    "category_id": 1, // 1 = occupied
    "bbox": [x, y, w, h], // Esquina superior izquierda, ancho y alto
    "segmentation": [[x1, y1, x2, y2, ...]], // Polígono de la plaza
    "car_id": 500 // Vehículo que la ocupa (-1: ninguno, -2: algo que no es un coche)
  }]
}
```

### Etiquetas disponibles

Para cada fichero de anotaciones se muestran sus secciones, sus categorías y cuántas anotaciones hay de cada una.

In [ ]:
annotations_glob = os.path.join(CONFIG['dataset']['source']['path'], 'annotations', '**', '*.json')
annotation_files = sorted(glob.glob(annotations_glob, recursive=True))

if not annotation_files:
    raise FileNotFoundError(f"No annotation files found: {annotations_glob}")

for path in annotation_files:
    with open(path) as f:
        coco = json.load(f)

    category_names = {c['id']: c['name'] for c in coco['categories']}
    counts = Counter(category_names[a['category_id']] for a in coco['annotations'])

    print(os.path.relpath(path, CONFIG['dataset']['source']['path']))
    print(f"  sections:    {list(coco.keys())}")
    print(f"  images:      {len(coco['images'])}")
    print(f"  categories:  {category_names}")
    print(f"  annotations: {dict(counts)}")